# 4a · Chunk & embed the manuals — SOLUTION

In [ ]:
import sys, os, re
from pathlib import Path
sys.path.insert(0, os.path.abspath("../.."))
from shared.llm import embed
import chromadb

manuals = sorted(Path("../../data/manuals").glob("*.md"))

def chunk_markdown(text, source):
    parts = re.split(r"\n(?=#{1,3}\s)", text)
    return [{"text": p.strip(), "source": source} for p in parts if len(p.strip()) >= 40]

chunks = []
for m in manuals:
    chunks += chunk_markdown(m.read_text(), m.name)
print(f"{len(chunks)} chunks")

In [ ]:
vectors = embed([c["text"] for c in chunks])
client = chromadb.EphemeralClient()
collection = client.create_collection("orbital_manuals")
collection.add(
    ids=[f"c{i}" for i in range(len(chunks))],
    embeddings=vectors,
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"]} for c in chunks],
)
print("stored:", collection.count())

In [ ]:
qv = embed("oxygen dropping")[0]
res = collection.query(query_embeddings=[qv], n_results=2)
for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
    print(meta["source"], "->", doc[:80])